In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random
from urllib.parse import urljoin

In [14]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36"
}

SLEEP = (0.5, 1.2)
TOTAL_TARGET = 1000

ART_RE = {
    "Detik.com": re.compile(r"/d-\d+/"),
    "Liputan6.com": re.compile(r"/read/\d+/"),
}


In [15]:
SOURCES = {
    "Liga Inggris": {
        "Liputan6.com": ["https://www.liputan6.com/tag/liga-inggris"],
        "Detik.com": ["https://sport.detik.com/sepakbola/liga-inggris"],
    },
    "Liga Italia": {
        "Liputan6.com": [
            "https://www.liputan6.com/tag/liga-italia",
            "https://www.liputan6.com/tag/serie-a",
            "https://www.liputan6.com/tag/juventus",
            "https://www.liputan6.com/tag/inter-milan",
            "https://www.liputan6.com/tag/ac-milan",
            "https://www.liputan6.com/tag/napoli",
            "https://www.liputan6.com/tag/as-roma",
        ],
        "Detik.com": ["https://sport.detik.com/sepakbola/liga-italia"],
    },
    "Liga Spanyol": {
        "Liputan6.com": ["https://www.liputan6.com/tag/liga-spanyol"],
        "Detik.com": ["https://sport.detik.com/sepakbola/liga-spanyol"],
    },
    "Liga Indonesia": {
        "Liputan6.com": [
            "https://www.liputan6.com/tag/bri-liga-1",
            "https://www.liputan6.com/tag/liga-indonesia",
        ],
        "Detik.com": ["https://sport.detik.com/sepakbola/liga-indonesia"],
    },
    "Olahraga Non Sepak Bola": {
        "Liputan6.com": [
            "https://www.liputan6.com/tag/bulutangkis",
            "https://www.liputan6.com/tag/motogp",
            "https://www.liputan6.com/tag/nba",
            "https://www.liputan6.com/tag/formula-1",
        ],
        "Detik.com": [
            "https://sport.detik.com/raket",
            "https://sport.detik.com/moto-gp",
            "https://sport.detik.com/basket",
            "https://sport.detik.com/sport-lain",
        ],
    },
}

In [16]:
def make_index_urls(base_url, media, page):
    if media == "Liputan6.com":
        return [
            f"{base_url}?page={page}",
            base_url if page == 1 else None,
        ]

    return [
        f"{base_url}/indeks/{page}",
        f"{base_url}/indeks?page={page}",
        f"{base_url}?page={page}",
        base_url if page == 1 else None,
    ]

def collect_urls_from_index(base_url, media, max_pages=80, needed=120, stop_after_empty=5):
    pattern = ART_RE[media]
    urls, seen = [], set()
    empty_pages = 0

    for page in range(1, max_pages + 1):
        page_found = 0

        for url in filter(None, make_index_urls(base_url, media, page)):
            try:
                r = requests.get(url, headers=HEADERS, timeout=20)
                if r.status_code >= 400:
                    continue
                soup = BeautifulSoup(r.text, "html.parser")
            except Exception:
                continue

            links = []
            if media == "Liputan6.com":
                articles = soup.find_all("article", class_="articles--iridescent-list--text-item")
                for art in articles:
                    a_tag = art.find("a", href=True)
                    if a_tag:
                        links.append(a_tag)
            else:
                links = soup.find_all("a", href=True)

            for a in links:
                link = urljoin(url, a["href"]).split("?")[0].split("#")[0]

                if pattern.search(link) and link not in seen:
                    seen.add(link)
                    urls.append(link)
                    page_found += 1

                    if len(urls) >= needed:
                        return urls

        empty_pages = empty_pages + 1 if page_found == 0 else 0
        if empty_pages >= stop_after_empty:
            break

        time.sleep(random.uniform(*SLEEP))

    return urls

def parse_article(url, media):
    target_url = url + "?page=all" if media == "Liputan6.com" else url

    try:
        r = requests.get(target_url, headers=HEADERS, timeout=20)
        if r.status_code >= 400:
            return None
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception:
        return None

    body_tags = []

    if media == "Liputan6.com":
        content_container = soup.find("div", class_="article-content-body__item-content")
        if content_container:
            body_tags = content_container.find_all("p")

    elif media == "Detik.com":
        content_container = soup.find("div", class_="detail__body-text")
        if content_container:
            body_tags = content_container.find_all("p")
    else:
        return None

    if not body_tags:
        return None

    text = "\n".join(p.get_text(" ", strip=True) for p in body_tags)
    text = re.sub(r"\s+", " ", text).strip()

    if len(text) < 200:
        return None

    return text



In [17]:
def build_dataset(sources, total_target=1000, max_pages=80, fill_media="Detik.com"):
    rows = []
    global_seen = set()
    per_category = total_target // len(sources)
    per_media_category = per_category // 2

    def fetch_from_urls(label, media, urls, target):
        taken = 0
        print(f"Fetching from {media}...", flush=True)

        for url in urls:
            if taken >= target:
                break

            if url in global_seen:
                continue

            text = parse_article(url, media)
            if text is None:
                continue

            global_seen.add(url)
            rows.append({
                "Teks": text,
                "Media": media,
                "Label": label,
            })

            taken += 1
            if taken % 10 == 0:
                print(f"{label} - {media}: {taken}/{target}", flush=True)

            time.sleep(random.uniform(*SLEEP))

        print(f"{media}: {taken}/{target} artikel", flush=True)
        return taken

    for label, source_group in sources.items():
        print(f"\n=== {label} ===", flush=True)
        collected_by_media = {}

        for media, index_urls in source_group.items():
            collected, seen_urls = [], set()
            print(f"Collecting {media} URLs...", flush=True)

            for index_url in index_urls:
                remaining = max(per_media_category * 3 - len(collected), per_media_category)
                new_urls = collect_urls_from_index(
                    index_url,
                    media,
                    max_pages=max_pages,
                    needed=remaining,
                )

                for article_url in new_urls:
                    if article_url not in seen_urls:
                        seen_urls.add(article_url)
                        collected.append(article_url)

                if len(collected) >= per_media_category * 3:
                    break

            collected_by_media[media] = collected
            print(f"{media} URLs found: {len(collected)}", flush=True)
            fetch_from_urls(label, media, collected, per_media_category)

        current = len([row for row in rows if row["Label"] == label])
        if current < per_category and fill_media in collected_by_media:
            shortfall = per_category - current
            print(f"Kategori {label} kurang {shortfall}, tambah dari {fill_media}...", flush=True)
            fetch_from_urls(label, fill_media, collected_by_media[fill_media], shortfall)
            current = len([row for row in rows if row["Label"] == label])

        if current < per_category:
            print(f"WARNING: {label} cuma dapat {current}/{per_category}. ", flush=True)

        print(f"Total {label}: {current}/{per_category}", flush=True)

    return pd.DataFrame(rows, columns=["Teks", "Media", "Label"])

In [18]:
df = build_dataset(SOURCES, total_target=1000, max_pages=80)

print("\n=== FINAL DATASET SUMMARY ===")
print(df.groupby(["Label", "Media"]).size())
print(f"Total rows: {len(df)}")

df.to_csv("dataset_berita_1000.csv", index=False, encoding="utf-8-sig")
print("Saved to dataset_berita_1000.csv")


=== Liga Inggris ===
Liputan6.com URLs found: 300
Fetching from Liputan6.com...
Liga Inggris - Liputan6.com: 10/100
Liga Inggris - Liputan6.com: 20/100
Liga Inggris - Liputan6.com: 30/100
Liga Inggris - Liputan6.com: 40/100
Liga Inggris - Liputan6.com: 50/100
Liga Inggris - Liputan6.com: 60/100
Liga Inggris - Liputan6.com: 70/100
Liga Inggris - Liputan6.com: 80/100
Liga Inggris - Liputan6.com: 90/100
Liga Inggris - Liputan6.com: 100/100
Liputan6.com: 100/100 artikel
Detik.com URLs found: 300
Fetching from Detik.com...
Liga Inggris - Detik.com: 10/100
Liga Inggris - Detik.com: 20/100
Liga Inggris - Detik.com: 30/100
Liga Inggris - Detik.com: 40/100
Liga Inggris - Detik.com: 50/100
Liga Inggris - Detik.com: 60/100
Liga Inggris - Detik.com: 70/100
Liga Inggris - Detik.com: 80/100
Liga Inggris - Detik.com: 90/100
Liga Inggris - Detik.com: 100/100
Detik.com: 100/100 artikel
Total Liga Inggris: 200/200

=== Liga Italia ===
Liputan6.com URLs found: 300
Fetching from Liputan6.com...
Liga Ital

In [19]:
df.head()

,Teks,Media,Label
0,"Liputan6.com, Jakarta - Mantan pelatih Liverpo...",Liputan6.com,Liga Inggris
1,"Liputan6.com, Jakarta - Manchester United kini...",Liputan6.com,Liga Inggris
2,"Liputan6.com, Jakarta - Persaingan untuk menda...",Liputan6.com,Liga Inggris
3,"Liputan6.com, Jakarta - Manchester United berp...",Liputan6.com,Liga Inggris
4,"Liputan6.com, Jakarta - Manchester United masi...",Liputan6.com,Liga Inggris
